In [11]:
import cv2 as cv
from cv2 import aruco
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_100)
parameters = aruco.DetectorParameters()
detector = aruco.ArucoDetector(aruco_dict, parameters)

img_aruco = cv.imread('dev_pictures/aruco_pic/aruco_0.png')

img_aruco_rotated = cv.rotate(img_aruco, 2)

SRC_COORDS = np.array([[[0., 0.], [60., 0.], [60., 60.], [0., 60.],
                        [0., 222.], [60., 222.], [60., 282.], [0., 282.]]], dtype=np.float32)

# Marker detektieren
corners, ids, rejected = detector.detectMarkers(img_aruco_rotated)


dstPoints = np.concatenate(corners, axis=1)
H, _ = cv.findHomography(srcPoints=SRC_COORDS, dstPoints=dstPoints, method=0)
H_inv = np.linalg.inv(H)

pts1 = np.float32([
    corners[1][0][0],  # oben-links
    corners[1][0][1],  # oben-rechts
    corners[0][0][3],  # unten-links
    corners[0][0][2],  # unten-rechts
])

pts1_reshaped = pts1.astype(np.float32).reshape(-1, 1, 2)

# Pixel in Bild zu welt
world_frame = cv.perspectiveTransform(pts1_reshaped, H_inv)

offset_raw = np.array([
    [-6, +6],
    [+6, +6],
    [-6, -6],
    [+6, -6]
], dtype=np.float32)

offset = offset_raw.reshape(-1, 1, 2)
pts1_2 = world_frame + offset
pts1_2_pixel = cv.perspectiveTransform(pts1_2, H)

min_x = np.min(pts1_2_pixel[:, 0, 0])
max_x = np.max(pts1_2_pixel[:, 0, 0])
min_y = np.min(pts1_2_pixel[:, 0, 1])
max_y = np.max(pts1_2_pixel[:, 0, 1])

width = int(max_x - min_x)
height = int(max_y - min_y)

pts2_proportional = np.float32([
    [0, 0],
    [width, 0],
    [0, height],
    [width, height]
])
M_warped = cv.getPerspectiveTransform(pts1_2_pixel, pts2_proportional)

img_warped = cv.warpPerspective(img_aruco_rotated, M_warped, (width, height))


corners_warped, ids, rejected = detector.detectMarkers(img_warped)
dstPoints_warped = np.concatenate(corners, axis=1)

H_warped, status = cv.findHomography(srcPoints=SRC_COORDS, dstPoints=dstPoints_warped, method=0)

H_inv_warped = np.linalg.inv(H)
